# **응급상황 자동 인식 및 응급실 연계 서비스**
# **단계1 : 응급상황 음성 인식 및 요약**

* 음성인식 : STT(Speech-to-Text)
    * 사용 모델 : OpenAI의 **Whisper-large-v3**
    * 제공받은 음성 파일과 새로 제작하는 5건 이상의 음성파일을 텍스트로 변환하고, 변환작업이 잘 되는지 확인해 봅시다.

* 텍스트 요약 및 핵심 키워드 도출
    * 사용 모델 : EXAONE-3.5 7B





#### 1) 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/KT_aivle/미니프로젝트/2024.11.18_미니프로젝트_6차_4일차_실습자료/'

### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

In [ ]:
# 경로 : /content/drive/MyDrive/project6_2/requirements.txt
# 경로가 다른 경우 아래 코드의 경로 부분을 수정하세요.

!pip install -r /content/drive/MyDrive/KT_aivle/미니프로젝트/2024.11.18_미니프로젝트_6차_4일차_실습자료/requirements.txt

#### 2) 라이브러리 로딩

In [ ]:
#필요한 라이브러리 설치 및 불러우기
import os
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
import openai
from openai import OpenAI
import json

# 더 필요한 라이브러리 추가 -------------
import torch
from datasets import load_dataset
from transformers import pipeline
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline, AutoModelForCausalLM, AutoTokenizer
import librosa

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

### (1) 제공된 데이터 변환
* 세부사항
    * 사용 모델 : whisper-large-v3
    * 제공 받은 오디오 파일을 읽어서 텍스트로 변환시켜 봅시다.
        * 반복문을 통해 파일 하나씩 읽어서 텍스트 변환
        * 변환된 텍스트를 데이터 프레임에 추가

|filename|text|
|----|----|
|audio3.mp3|어쩌구 저쩌구...급해요.|

* 음성파일 변환

In [ ]:
# 음성파일 경로 지정
audio_path = '/content/drive/MyDrive/KT_aivle/미니프로젝트/2024.11.18_미니프로젝트_6차_4일차_실습자료/audio/'

In [ ]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline

processor = AutoProcessor.from_pretrained("openai/whisper-large-v3-turbo")
model = AutoModelForSpeechSeq2Seq.from_pretrained("openai/whisper-large-v3-turbo")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.71M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

In [ ]:
import torch
from datasets import load_dataset
from transformers import pipeline

In [ ]:
#오디오 30초 자르는건 선넘네
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

result = pipe(sample, return_timestamps=True, generate_kwargs={"language": "korean"})
print(result["text"])

In [ ]:
# numpy 배열로 변환하는 코드
import librosa
filename = 'audio2.mp3'
audio_file = open(audio_path + filename, "rb")
audio_array, sampling_rate = librosa.load(audio_file, sr=None)
result = pipe({"array": audio_array, "sampling_rate": sampling_rate}, return_timestamps=True)
print(result["text"])

/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


 119조 제가 지금 열이 열이 올랐어요. 몇 도냐면은 38도 정도 돼요. 머리가 아프고 좀 띵한 것 같아요. 오하니 좀 들어요. 어떻게 해야 할까요?


* 음성파일 변환 함수 생성

In [ ]:
audio_path = '/content/drive/MyDrive/KT_aivle/미니프로젝트/2024.11.18_미니프로젝트_6차_4일차_실습자료/test_audio/audio/'

In [ ]:
def audio_to_text(audio_path, filename):
    # 오디오 파일을 읽어서, 위스퍼를 사용한 변환
    audio_file = open(audio_path + filename, "rb")
    audio_array, sampling_rate = librosa.load(audio_file, sr=None)
    result = pipe({"array": audio_array, "sampling_rate": sampling_rate}, return_timestamps=True)

    # 결과 반환
    print(result["text"])
    return result

In [ ]:
# 음성파일 이름을 리스트에 담기
file_names = [f for f in os.listdir(audio_path) if os.path.isfile(os.path.join(audio_path, f))]
print(file_names)

In [ ]:
# 반복문을 통해, 파일 하나씩 읽어서 텍스트 변환, 변환된 텍스트를 데이터 프레임에 추가

# 빈 데이터프레임 선언
df = pd.DataFrame(columns=['file_name','transcription'])

# 반복문 수행하면서 오디오 변환
for i, file_name in enumerate(file_names):
    transcript = audio_to_text(audio_path, file_name)
    df.loc[len(df)] = {'file_name': file_name, 'transcription': transcript}

# 데이터프레임 결과 조회
print(df)

 네, 119입니다. 말씀하세요. 아 네, 그 계속 토로 해가지고. 어느 분. 어 어 병원은 지금 움직일 수가 없어서. 어느 분이 계속 토로 하세요? 아 저희 언니요. 언니가요? 네. 토하시고. 네. 우리 쪽이랑 호흡은 다 있어요? 네. 아 네, 보급차 보내드릴 건데 주소가 어떻게 되세요? 강서구. 네. 화공로 18길. 잠시만요. 화공 18길. 강서 뉴 타워. 잠시만요. 강서 뉴 타워. 네. 아 잠시만요. 언니분이랑 동생은 최근에 뭐 코로나 관련해서 의심 증상이나 뭐 접속했다고 문자 받으신 거 없으시죠? 아 네 알겠습니다. 배원도 나가면서 전화드릴 수 있으니까 전화 한번 안내 좀 부탁드릴게요. 아 네 알겠습니다.
 감사합니다. 무슨 일이세요? 아 네 여보세요? 네 말씀하세요. 무슨 일이세요? 아 지금 뭐 현압을 해보니까 현압이 너무 높아요. 누구야 선생님이요? 네 177이 넘고 지금 막 그 약간 말이 안 나오고 춥고 막 그러거든요. 조금만 보내드릴까요 선생님? 네 여기 아우 힘들어요. 말이 어눌하신 거예요? 말이 하는 게 어려운 거예요? 아 추워요 지금. 그냥 추운 거예요? 말이 어누라 그냥 한쪽으로 힘 빠지는 건 없으시죠? 네 추워요. 네 구급차 위치 좀 보내드릴 구급차 좀 보내드릴게요. 위치 좀 알려주실 수 있으실까요? 네 여기 포르주아 아파트. 하곡 포르주아 맞으세요? 선생님? 네. 맞으세요? 네 네. 혹시 근데 코로나 관련해서 뭐 확진자를 접촉하거나 그러신 거 있으세요? 아니요. 코로나 주사를 제가 화요일 날 맞았어요. 네. 아 그러는데 한 3일만은 괜찮았는데 어제서부터 약간 하락이 올라가더라고요. 선생님 그쪽으로 하구 프로지오 네. 네. 네. 네. 네. 네. 네. 네. 네. 네. 네. 혹시 모릅니다. 문 좀 열어두세요. 선생님. 전화 구급회원들 가면서 전화할 거고요. 전화 잘 받아주세요. 알겠습니다. 감사합니다.
 여보세요? 예예. 왜 애가 갑자기 지금 시선을 이렇게 며이셔죠? 빨리 와주세요. 어디로 가요? 여기 송방어 오금로? 오금로 36길 36길에

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


 네 그 어 여기가 보문동 자이 아파트거든요. 보문 파크뷰 자이요? 네. 여기 할아버지가 지금 쓰러지셔 가지고. 집 안에서요? 네. 몇 동 몇 호예요? 네 되세요? 하루가 돼. 응세가. 일. 네 움직임 전혀 없으시고요. 음 의료지도 제가 연결할게요. 의료진한테 안내 좀 해주세요.


Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


 1월 1일부터. 아 네 저기 지금 요양원이에요. 네 근데 그 어르신께서 지금 갑자기 아침에 질출혈을 좀 많이 하셨거든요. 할머니 질. 예 할머니 예 10시부터 해가지고. 그래서 출혈량이 있어가지고요. 병원 가시려고요? 예 지금 보호자 연락했는데 출혈이 많고 혈압이 지금 떨어져요. 계속. 네, 50 있어요? 지금? 예, 지금은 있어요. 60 근데 지금 혈압 60에 40이에요. 예. 혈압이 60에 40이라고요? 네, 지금요. 막. 아, 지금 얼른 보낼게요. 주소 알려주세요. 예, 수색동이요. 네네. 수색 실버케요. 수색 실버케요? 몇 층으로 가면 돼요? 아, 1층으로 오시면은 저 갈게요. 네. 따로 거기에 확진자 있거나 그런 거 없어요? 정리 사항이. 아 저희 그런 거 없어요. 네 없어요. 지금 차는 보냈는데요. 네, 네. 어 집에서 나오는 게 맞아요? 대변 쪽에서 나오는 거 아니고? 아 집에서 집에서요. 나오는 거예요? 어저께 근데 그 2차 백신 맞으셨거든요? 아 네, 네. 네, 네. 차 보냈고 그 수색역 쪽에서 구급차 어 수색역에서 구급차가 없어가지고 조금 멀리서 가요. 소대문서가 저쪽 한 5km 떨어진 데서 가요. 연희대에서. 아 네 네 알겠습니다.


...

 네 수고하세요. 여보세요? 네 저 119 성함입니다. 무슨 일이세요? 저기 여기 요구리스타워에요. 네. 저기 그 오줌이 안 나와서 응급비를 좀 가야 하겠는데. 선생님 본인이 소변이 안 나온다고요? 예 소변이 잘 나와서 며칠 전에도 1번 갔었는데. 어떻게? 노블레스 타워 1차 네 맞아요 일단 그쪽으로 구급자 전 보냈고요 예예 환자분 감기 증상이 있거나 코로나 확진자는 아니에요? 감기 증상이 있거나 코로나 확진자는 아니에요? 그건 아니에요 일단 구급자 전 보냈습니다 만나보세요 전입변 2대를 수술했는데 그 후유증이에요 전차 보내는 사람이고요 출동한 구급자한테 환자분 과거력이 돼서 말씀해 주세요 저는 출동한 구급배원이 아니라 차를 보내낸 사람이니까 출동한 구급배원이랑 병원 선정이랑 환자분 상태에 대해서 말씀 나누세요.
 아 여보세요? 저기 지금 역전을 해내거든요. 여기 술을 마신 애가 토를 했는데 정신을 못 차려요. 지금 그분이 응급실로 가셔야 되는 거예요? 기가 도움이 필요하신 거예요? 지금 그냥 정신이 없어요. 정신이 없이 쓰러져 있어요? 네 그냥 진짜 아예 정신이 없어요. 네 그냥 토를 하고 아무것도 못 해요. 신고자분 거기 위치 좀 알려줘요. 무슨 어디라고 어디 앞이라고 하셨죠? 신림 사거리 역전 할메요. 역전 할메요? 네 지금 빨리 오셔야 될 것 같아요. 신림 할메 차 가고 있으니까 잠시만 볼게요. 역전 할메. 지금 눈에 초점이 없거든요. 예 알겠습니다. 신림 할메 술 너무 많이 먹으면 그럴 수 있으니까 너무. 그렇습니다. 네 지금 근데 누워있거든요. 요. 편하게 누워 편하게 누워있게 해주세요. 손나 잘 쉬게 누워있게 해주시고. 컷 때문에 입이 막혔어요. 이 이 숨 쉬게 입 벌려주세요. 숨 잘 쉬게 남자분이 여자분이에요. 남자분요. 남자분 숨 잘 못 쉬는 거 같아요? 끊지 마시고 차 출동했고요. 의료진 연결할 거니까 신고자분 신착하시고 신고자분 네. 신고자분 숨 잘 쉬게 의료진 연결할 거예요. 끊지 마세요.


Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


 1 9입니다. 말씀하세요. 네 저기 엄마가 많이 아프신데 그 혹시 코로나를 접종을 안 했었거든요. 그런데 이동은 가능한가요? 어 뭐 지금 병원 응급실 지금 혹시 뭐 열 같은 건 알아요? 어머니가 열이나 기침 이런 증상. 그런 건 아닌데. 없고 그냥 일단 복지도 안 맞았는데 어디가 아프신 거예요? 지금? 지금 몸살 비슷한 거 같은데. 네, 네. 저희 뭐 물도 잘 못 마시고. 예. 그러고 계시거든요. 누워만. 아 주소 알 수 있을까요? 음 잠시만요. 예, 예. 어 노원구. 예. 월계로 40차 가길. 잠시만요. 월계로 44 각일 네. 청구빌라 왔어요? 예. 청구빌라 맞습니다. 지금 같이 계신 거 아니에요? 선생님 그러면? 저는 저는 답심리에 있는데. 아 예예. 그러면 어머니 전화번호? 네. 네. 전화번호 알려주세요. 전화번호가 네. 네네. 아 네 알겠습니다. 제가 나가면서 전화 한번 해볼게요 그러면 어머니한테. 그러면은 제가요. 예예. 하나 오빠가 갈 수 있거든요. 예예. 그러면은 좀 알려드려도 될까요? 오빠 번호. 아 그럼 그냥 오빠 번호를 알려주세요. 그게 나을 것 같은데. 예. 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 네 

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


 네 119입니다. 말씀하세요. 어 여기 지나가는데. 네. 여기 증산역 뒷골목이거든요. 여기 가게가 뭐 부동산 있고 희망 부동산 있고. 아 잠시만요. 희망 부동산이요? 잠시만요. 아 금산구 증산동. 여기 1 4번 출구 3번 출구 요 뒤쪽 라인인 거죠? 네 희망 부동산 뒤에 횟집 있거든요. 희망 부동산 뒤쪽 횟집이요. 횟집 그 옆에 뭐 만두가게인가 노란색 집인데. 네. 그 그 문 앞에 사람이 쓰러져 있어요. 아 남자분이요? 여자분이에요? 남자분이요. 지나가다가 그냥 쓰러져 걸어가다가 그냥 쓰러졌다고 하던데. 아 그냥 걸어가서 앞에 쓰러졌대요? 예 검은색 옷 입고 검은 가방. 잠시만요. 젊은 사람 같아요. 혹시 의식이랑 호흡 있는지 확인 가능할까요? 아 저 제가 지금 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 차 

In [ ]:
df

,file_name,transcription
0,651e464d69a4f266f0626820_20220101.wav,"{'text': ' 네, 119입니다. 말씀하세요. 아 네, 그 계속 토로 해가지고..."
1,651e464d69a4f266f062681a_20220101.wav,{'text': ' 감사합니다. 무슨 일이세요? 아 네 여보세요? 네 말씀하세요. ...
2,651e464d69a4f266f062681b_20220101.wav,{'text': ' 여보세요? 예예. 왜 애가 갑자기 지금 시선을 이렇게 며이셔죠?...
3,651e464d69a4f266f062680e_20220101.wav,{'text': ' 네 여기 저 방산 센트럴 아이터크. 잠시만요. 방산 센트럴 아이...
4,651e464d69a4f266f0626815_20220101.wav,{'text': ' 네 그 어 여기가 보문동 자이 아파트거든요. 보문 파크뷰 자이요...
...,...,...
824,651e464d69a4f266f0626a16_20220101.wav,{'text': ' 불러 거거든? 차 키는 여기다 둔 거가? 어. 아 네 저기 저희...
825,651e464d69a4f266f0626a18_20220101.wav,"{'text': ' 네, 혹시 주이세요? 이 쇠나파 코로나 그 주사 맞고 아이가 지..."
826,651e464d69a4f266f0626a21_20220101.wav,{'text': ' 119입니다. 여보세요. 119입니다. 예 예 여기 농협인데요....
827,651e464d69a4f266f0626a22_20220101.wav,{'text': ' 여기 번일동 신진빌라 천천히 좀 불러주세요. 주소 좀. 번일동이...


In [ ]:
df.head(10)

,file_name,transcription
0,651e464d69a4f266f0626820_20220101.wav,"{'text': ' 네, 119입니다. 말씀하세요. 아 네, 그 계속 토로 해가지고..."
1,651e464d69a4f266f062681a_20220101.wav,{'text': ' 감사합니다. 무슨 일이세요? 아 네 여보세요? 네 말씀하세요. ...
2,651e464d69a4f266f062681b_20220101.wav,{'text': ' 여보세요? 예예. 왜 애가 갑자기 지금 시선을 이렇게 며이셔죠?...
3,651e464d69a4f266f062680e_20220101.wav,{'text': ' 네 여기 저 방산 센트럴 아이터크. 잠시만요. 방산 센트럴 아이...
4,651e464d69a4f266f0626815_20220101.wav,{'text': ' 네 그 어 여기가 보문동 자이 아파트거든요. 보문 파크뷰 자이요...
5,651e464d69a4f266f062682f_20220101.wav,{'text': ' 1월 1일부터. 아 네 저기 지금 요양원이에요. 네 근데 그 어...
6,651e464d69a4f266f0626869_20220101.wav,{'text': ' 여보세요? 예 선생님 119 입니다. 네네. 예 선생님 아 부모...
7,651e464d69a4f266f062680f_20220101.wav,{'text': ' 119입니다. 예 저 119죠? 네네. 아 여기 아저씨가 자리를...
8,651e464d69a4f266f0626819_20220101.wav,{'text': ' 119 입니다. 여보세요? 예. 예 여기 노인장반이 좀 저 쓰러...
9,651e464d69a4f266f062681f_20220101.wav,{'text': ' 네. 일로 봅니다. 아 네. 그 이거 제가 예 저거 운기차 좀 ...


In [ ]:
df.to_csv("dataset_whisper-3.csv")

### (2) 오디오 데이터 추가 수집(제작) 및 변환


* 문서 요약 함수로 생성

In [ ]:
#7.8는 응답속도가 너무 느려서 2.4로 줄여서 해야될듯

trnas_model_name = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

trnas_model = AutoModelForCausalLM.from_pretrained(
    trnas_model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(trnas_model_name)

# Choose your prompt
prompt = "Explain how wonderful you are"  # English example
prompt = "스스로를 자랑해 봐"       # Korean example

messages = [
    {"role": "system", "content": "You are EXAONE model from LG AI Research, a helpful assistant."},
    {"role": "user", "content": prompt}
]
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

output = trnas_model.generate(
    input_ids.to("cuda"),
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=512,
    do_sample=False,
)
print(tokenizer.decode(output[0]))

In [ ]:
def text_summary(input_text):
    # 시스템 역할과 응답 형식 지정
    system_role = "당신은 응급상황에서 어떠한 상황인지 파악하고, 증상을 간단한 키워드로 요약하는 어시스턴트입니다. 응답은 다음의 형식을 지켜주세요 {\"summary\": \"텍스트 요약\", \"부위\": \"다친 부위\", \"증상\": \"관련 증상\"}"

    # 입력데이터를 EXAONE-3.5에 전달하고 답변 받아오기
    messages = [
    {"role": "system", "content": system_role},
    {"role": "user", "content": input_text}]

    input_ids = tokenizer.apply_chat_template(messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt")

    output = trnas_model.generate(
    input_ids.to("cuda"),
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=256,
    do_sample=False,)

    # 응답 받기
    response = tokenizer.decode(output[0])

    match = re.search(r'json\n(.*?)\n', response, re.DOTALL)
    if match:
       json_str = match.group(1)
       print(json.loads(json_str))
    else: raise ValueError("JSON 데이터를 찾을 수 없습니다.")
    # 응답형식을 정리하고 return
    return json.loads(json_str)

In [ ]:
'''
# 기본 포맷
# 시스템 역할과 응답 형식 지정
system_role = "당신은 응급상황에서 어떠한 상황인지 파악하고, 증상을 간단한 키워드로 요약하는 어시스턴트입니다. 응답은 다음의 형식을 지켜주세요 {\"summary\": \"텍스트 요약\", \"부위\": \"다친 부위\", \"증상\": \"관련 증상\"}"
input_text = "지금 아빠가 넘어졌어요. 머리에서 피가 나는데 숨은 쉬고 있어요. 지금 막 일어났어요. 근데 조금 어지럽다고 하네요. 네, 네. 계단에서 굴렀어요. 지금은 물 마시고 있는데 이거 응급실로 가봐야 할까요? 피도 지금 멎었어요. 네, 네. 나이는 49살이세요. 어떻게 해야 할지 모르겠어요."
# 입력데이터를 EXAONE-3.5에 전달하고 답변 받아오기
messages = [
{"role": "system", "content": system_role},
{"role": "user", "content": input_text}]

input_ids = tokenizer.apply_chat_template(messages,
tokenize=True,
add_generation_prompt=True,
return_tensors="pt")

output = trnas_model.generate(
input_ids.to("cuda"),
eos_token_id=tokenizer.eos_token_id,
max_new_tokens=128,
do_sample=False,)

# 응답 받기
# print(tokenizer.decode(output[0]))
response = tokenizer.decode(output[0])
match = re.search(r'json\n(.*?)\n', response, re.DOTALL)
if match:
    json_str = match.group(1)
    print(json.loads(json_str))
else: raise ValueError("JSON 데이터를 찾을 수 없습니다.")
print(json.loads(json_str))
'''

'\n# 기본 포맷\n# 시스템 역할과 응답 형식 지정\nsystem_role = "당신은 응급상황에서 어떠한 상황인지 파악하고, 증상을 간단한 키워드로 요약하는 어시스턴트입니다. 응답은 다음의 형식을 지켜주세요 {"summary": "텍스트 요약", "부위": "다친 부위", "증상": "관련 증상"}"\ninput_text = "지금 아빠가 넘어졌어요. 머리에서 피가 나는데 숨은 쉬고 있어요. 지금 막 일어났어요. 근데 조금 어지럽다고 하네요. 네, 네. 계단에서 굴렀어요. 지금은 물 마시고 있는데 이거 응급실로 가봐야 할까요? 피도 지금 멎었어요. 네, 네. 나이는 49살이세요. 어떻게 해야 할지 모르겠어요."\n# 입력데이터를 EXAONE-3.5에 전달하고 답변 받아오기\nmessages = [\n{"role": "system", "content": system_role},\n{"role": "user", "content": input_text}]\n\ninput_ids = tokenizer.apply_chat_template(messages,\ntokenize=True,\nadd_generation_prompt=True,\nreturn_tensors="pt")\n\noutput = trnas_model.generate(\ninput_ids.to("cuda"),\neos_token_id=tokenizer.eos_token_id,\nmax_new_tokens=128,\ndo_sample=False,)\n\n# 응답 받기\n# print(tokenizer.decode(output[0]))\nresponse = tokenizer.decode(output[0])\nmatch = re.search(r\'json\n(.*?)\n\', response, re.DOTALL)\nif match:\n    json_str = match.group(1)\n    print(json.loads(json_str))\nelse: raise ValueError("JSON 데이터를

* 저장된 text를 하나씩 불러와서 요약하고 다시 저장하기

In [ ]:
import re
response = []
for i in range(len(df)):
    input_text = df['transcription'][i]['text']
    print(input_text)
    summary = text_summary(input_text)
    response.append(summary['summary'])

 네, 119입니다. 말씀하세요. 아 네, 그 계속 토로 해가지고. 어느 분. 어 어 병원은 지금 움직일 수가 없어서. 어느 분이 계속 토로 하세요? 아 저희 언니요. 언니가요? 네. 토하시고. 네. 우리 쪽이랑 호흡은 다 있어요? 네. 아 네, 보급차 보내드릴 건데 주소가 어떻게 되세요? 강서구. 네. 화공로 18길. 잠시만요. 화공 18길. 강서 뉴 타워. 잠시만요. 강서 뉴 타워. 네. 아 잠시만요. 언니분이랑 동생은 최근에 뭐 코로나 관련해서 의심 증상이나 뭐 접속했다고 문자 받으신 거 없으시죠? 아 네 알겠습니다. 배원도 나가면서 전화드릴 수 있으니까 전화 한번 안내 좀 부탁드릴게요. 아 네 알겠습니다.
{'summary': '언니가 구토 증상을 보이고 있으며, 호흡에는 문제가 없음.', '부위': '위장', '증상': '구토'}
 감사합니다. 무슨 일이세요? 아 네 여보세요? 네 말씀하세요. 무슨 일이세요? 아 지금 뭐 현압을 해보니까 현압이 너무 높아요. 누구야 선생님이요? 네 177이 넘고 지금 막 그 약간 말이 안 나오고 춥고 막 그러거든요. 조금만 보내드릴까요 선생님? 네 여기 아우 힘들어요. 말이 어눌하신 거예요? 말이 하는 게 어려운 거예요? 아 추워요 지금. 그냥 추운 거예요? 말이 어누라 그냥 한쪽으로 힘 빠지는 건 없으시죠? 네 추워요. 네 구급차 위치 좀 보내드릴 구급차 좀 보내드릴게요. 위치 좀 알려주실 수 있으실까요? 네 여기 포르주아 아파트. 하곡 포르주아 맞으세요? 선생님? 네. 맞으세요? 네 네. 혹시 근데 코로나 관련해서 뭐 확진자를 접촉하거나 그러신 거 있으세요? 아니요. 코로나 주사를 제가 화요일 날 맞았어요. 네. 아 그러는데 한 3일만은 괜찮았는데 어제서부터 약간 하락이 올라가더라고요. 선생님 그쪽으로 하구 프로지오 네. 네. 네. 네. 네. 네. 네. 네. 네. 네. 네. 혹시 모릅니다. 문 좀 열어두세요. 선생님. 전화 구급회원들 가면서 전화할 거고요. 전화 잘 받아주세요. 알겠습니다. 감사합니다.


KeyboardInterrupt: 

In [ ]:
for i in range(len(df)):
    try:
        input_text = df['transcription'][i]['text']  # 'text' 필드 확인
        if not isinstance(input_text, str):
            raise ValueError(f"Invalid input format at index {i}: {input_text}")

        summary = text_summary(input_text)
        response.append(summary)
    except Exception as e:
        print(f"Error at index {i}: {e}")
        response.append(None)

{'summary': '언니가 구토 증상을 보이고 있으며, 호흡에는 문제가 없음.', '부위': '위장', '증상': '구토'}
{'summary': '갑작스러운 고혈압 상승, 말 어눌함, 추위 느낌, 약간의 의식 저하 의심', '부위': '전신', '증상': '고혈압 (177 이상), 말 어눌함, 추위 느낌, 의식 저하 의심'}
{'summary': '성인 남녀가 갑자기 의식을 잃고 눈동자가 고정된 상태로 바닥에 쓰러졌으며, 호흡은 있으나 불규칙적일 가능성이 있음.', '부위': '전신', '증상': '의식 상실, 고정된 눈동자, 호흡 있음 (불규칙 가능성)'}
{'summary': '집사람이 복부에서 탈장 증상을 보이고 있습니다.', '부위': '복부', '증상': '탈장'}
{'summary': '할아버지가 집 안에서 쓰러져 움직이지 않고 있습니다.', '부위': '전신', '증상': '의식 불명, 무반응'}
Error at index 5: Expecting property name enclosed in double quotes: line 1 column 80 (char 79)
{'summary': '사망 의심되는 환자, 과거 며칠 전 사망 추정', '부위': '전신', '증상': '의식 상실, 호흡 없음'}
{'summary': '남편이 계단에서 떨어져 발목 부상 의심, 다리 굽힘 증상 지속', '부위': '발목', '증상': '다리 굽힘, 통증'}
{'summary': '남성 노인이 왼쪽 골반 골절로 쓰러졌으며, 코로나 관련 증상은 없음.', '부위': '왼쪽 골반', '증상': '골절, 쓰러짐'}
{'summary': '아버지가 호흡곤란을 겪고 있으며, 현재 구급차가 도착한 상태입니다.', '부위': '전체', '증상': '호흡곤란'}
{'summary': '아랫배 통증이 시작되어 점점 심해지고 있음', '부위': '아랫배', '증상': '지속적인 복통'}
{'summary': '심한 복통 호소', '부위': '복부', '증상': '복통'}
{'su

In [ ]:
response

['언니가 구토 증상을 보이고 있으며, 호흡에는 문제가 없음.',
 {'summary': '언니가 구토 증상을 보이고 있으며, 호흡에는 문제가 없음.', '부위': '위장', '증상': '구토'},
 {'summary': '갑작스러운 고혈압 상승, 말 어눌함, 추위 느낌, 약간의 의식 저하 의심',
  '부위': '전신',
  '증상': '고혈압 (177 이상), 말 어눌함, 추위 느낌, 의식 저하 의심'},
 {'summary': '성인 남녀가 갑자기 의식을 잃고 눈동자가 고정된 상태로 바닥에 쓰러졌으며, 호흡은 있으나 불규칙적일 가능성이 있음.',
  '부위': '전신',
  '증상': '의식 상실, 고정된 눈동자, 호흡 있음 (불규칙 가능성)'},
 {'summary': '집사람이 복부에서 탈장 증상을 보이고 있습니다.', '부위': '복부', '증상': '탈장'},
 {'summary': '할아버지가 집 안에서 쓰러져 움직이지 않고 있습니다.', '부위': '전신', '증상': '의식 불명, 무반응'},
 None,
 {'summary': '사망 의심되는 환자, 과거 며칠 전 사망 추정', '부위': '전신', '증상': '의식 상실, 호흡 없음'},
 {'summary': '남편이 계단에서 떨어져 발목 부상 의심, 다리 굽힘 증상 지속',
  '부위': '발목',
  '증상': '다리 굽힘, 통증'},
 {'summary': '남성 노인이 왼쪽 골반 골절로 쓰러졌으며, 코로나 관련 증상은 없음.',
  '부위': '왼쪽 골반',
  '증상': '골절, 쓰러짐'},
 {'summary': '아버지가 호흡곤란을 겪고 있으며, 현재 구급차가 도착한 상태입니다.',
  '부위': '전체',
  '증상': '호흡곤란'},
 {'summary': '아랫배 통증이 시작되어 점점 심해지고 있음', '부위': '아랫배', '증상': '지속적인 복통'},
 {'summary': '심한 복통 호소', '부위': '복부', '증상': '복통'},
 {'summary': '

In [ ]:
response_df = pd.DataFrame({'summary':response})
response_df

,summary
0,"언니가 구토 증상을 보이고 있으며, 호흡에는 문제가 없음."
1,"{'summary': '언니가 구토 증상을 보이고 있으며, 호흡에는 문제가 없음.'..."
2,"{'summary': '갑작스러운 고혈압 상승, 말 어눌함, 추위 느낌, 약간의 의..."
3,{'summary': '성인 남녀가 갑자기 의식을 잃고 눈동자가 고정된 상태로 바닥...
4,"{'summary': '집사람이 복부에서 탈장 증상을 보이고 있습니다.', '부위'..."
...,...
825,"{'summary': '급체로 인한 복통 호소, 의식과 호흡 유지 중', '부위':..."
826,"{'summary': '백신 접종 후 아이가 근육통을 호소하고 있으며, 현재 증상이..."
827,"{'summary': '노인이 농협 정문 근처에서 넘어져 일어나지 못함', '부위'..."
828,"{'summary': '아버지가 갑작스러운 오한, 발열, 두드러기 증상을 보이고 있..."


In [ ]:
response_df.to_csv("whisper-3_EXAONE_response.csv")

In [ ]:
label_class = pd.read_csv('/content/drive/MyDrive/KT_aivle/미니프로젝트/2024.11.18_미니프로젝트_6차_4일차_실습자료/2024.11.19_미니프로젝트 6차_5일차 실습자료/중증도 카테고리.csv', encoding='euc-kr')
label_class

,구분1,구분2,구분3,등급
0,물질오용,물질오용/중독,중증 호흡곤란,1
1,물질오용,물질오용/중독,쇼크,1
2,물질오용,물질오용/중독,무의식(GCS 3-8),1
3,물질오용,물질오용/중독,중등도 호흡곤란,2
4,물질오용,물질오용/중독,혈역학적 장애,2
...,...,...,...,...
2024,일반,경증 호소,만성 말초성 중증 통증(8-10),4
2025,일반,경증 호소,만성 중심성 경증 통증(<4),5
2026,일반,경증 호소,급성 말초성 경증 통증(<4),5
2027,일반,경증 호소,만성 말초성 통증(<8),5


In [ ]:
label_class = label_class.fillna("")

In [ ]:
# 등급 매기기 함수 정의
def assign_grade(response, label_class):
    response_symptoms = response['증상'].split(', ')
    # 등급 초기화
    assigned_grade = None

    # 각 증상에 대해 label_class에서 매칭된 등급 찾기
    for symptom in response_symptoms:
        matched_rows = label_class[label_class['구분3'].str.contains(symptom, na=False)]
        if not matched_rows.empty:
            # 현재까지 할당된 등급과 매칭된 등급 중 더 낮은 등급 선택
            grade = matched_rows['등급'].min()
            if assigned_grade is None or grade < assigned_grade:
                assigned_grade = grade

    # 등급이 없는 경우 '등급 없음'으로 설정
    response['등급'] = assigned_grade if assigned_grade is not None else '등급 없음'
    return response

# 분류 함수 정의
def classify_response(response_data, label_class):
    data_f = []
    for i in range(len(response_df)):
        # ai를 사용하여 응급 상황 요약 생성
        AI_response = text_summary(response_df['summary'][i])
        # 생성된 응답을 바탕으로 등급 매기기
        response_with_grade = assign_grade(AI_response, label_class)
        data_f.append(response_with_grade)

    # 데이터프레임으로 변환 및 저장
    data_df = pd.DataFrame(data_f)
    return data_df

response_data = response.copy()

# 분류 수행 및 결과 저장
classified_data = classify_response(response_df, label_class)
print(classified_data)

TypeError: can only concatenate str (not "dict") to str

### (2) 전국 병원 응급실 정보 수집



#### 데이터 수집

In [ ]:
# path 확인
path

'/content/drive/MyDrive/KT_aivle/미니프로젝트/2024.11.18_미니프로젝트_6차_4일차_실습자료/'

In [ ]:
# 응급실 데이터 수집하기
# 응급실 목록정보 조회

url = 'http://apis.data.go.kr/B552657/ErmctInfoInqireService/getEgytListInfoInqire'
serviceKey = ''    # 여러분의 일반 인증키(Decoding)

params = {
    'serviceKey': serviceKey,
    'pageNo': '1', 'numOfRows': '1000',  # 전체 응급실 수가 500여개 됨. 1000개면 충분
    'format': 'xml'
}

response = requests.get(url, params = params)

# 정상 수행 되었다면 200
print(response)

<Response [200]>


In [ ]:
# response xml에서 주요 정보 찾기
root = ET.fromstring(response.text)

data = []

for item in root. findall('.//item'):
    duty_name = item.findtext('dutyName') #기관명
    duty_addr = item.findtext('dutyAddr') #주소
    # 필요한 정보 추가
    latitude = item.findtext('wgs84Lat')
    longitude = item.findtext('wgs84Lon')

    # 빈 리스트 data에 딕시너리 형태({'칼럼이름':값, ...})로 저장(추가)
    data.append({'기관명' : duty_name, '기관주소': duty_addr, '위도' : latitude, '경도': longitude})

# 데이터프레임으로 변환
df_basic =  pd.DataFrame(data)

In [ ]:
df_basic

,기관명,기관주소,위도,경도
0,(의)내경의료재단울산제일병원,울산광역시 남구 남산로354번길 26 (신정동),35.54823820112527,129.30701143429678
1,(의)서일의료재단기장병원,부산광역시 기장군 기장읍 대청로72번길 6,35.23602946449906,129.21649161387128
2,(의)성세의료재단 뉴성민병원,"인천광역시 서구 칠천왕로33번길 17 (석남동, 신석로 70(석남1동, 성민병원))",37.5089936801167,126.669478649026
3,(의)영문의료재단다보스병원,"경기도 용인시 처인구 백옥대로1082번길 18, 다보스종합병원 (김량장동)",37.234641294534285,127.21049868474802
4,(의)효심의료재단용인서울병원,경기도 용인시 처인구 고림로 81 (고림동),37.240316373,127.2144914405
...,...,...,...,...
521,효산의료재단안양샘병원,"경기도 안양시 만안구 삼덕로 9 (안양동, 안양샘병원)",37.39340413136221,126.92447734066778
522,효산의료재단지샘병원,"경기도 군포시 군포로 591 (당동, (G샘병원)군포샘병원)",37.35864464913468,126.94735972779895
523,효성시티병원,부산광역시 해운대구 해운대로 135 (재송동),35.18541271514478,129.12145899191526
524,흑룡의원,인천광역시 옹진군 백령면 백령로 831,37.959524356,124.6654985764


In [ ]:
!pip install geopy

In [ ]:
from geopy.geocoders import Nominatim
import time

In [ ]:
geolocator = Nominatim(user_agent='chiricuto')

# 주소에서 위도와 경도를 찾는 함수
def get_coordinates(address):
    try:
        simple_address = address.split(",")[0].split("(")[0] # 주소에서 상세한 부분을 제거하여 더 단순화
        location = geolocator.geocode(simple_address)
        if location is not None:
            return location.latitude, location.longitude
        else:
            print(f"주소를 찾을 수 없습니다: {address}")
            return None, None
    except Exception as e:
        print(f"오류 발생: {e}, 주소: {address}")
        return None, None

# 데이터프레임에 위도와 경도 추가
for i, row in df_basic.iterrows():
    latitude, longitude = get_coordinates(row['기관주소'])
    df_basic.loc[i, '위도'] = latitude
    df_basic.loc[i, '경도'] = longitude
    time.sleep(1)

# 결과 출력
print(df_basic)

주소를 찾을 수 없습니다: 경기도 용인시 처인구 백옥대로1082번길 18, 다보스종합병원 (김량장동)


주소를 찾을 수 없습니다: 경상북도 김천시 모암길 24 (모암동)


주소를 찾을 수 없습니다: 전라남도 구례군 구례읍 동편제길 4
주소를 찾을 수 없습니다: 전라남도 순천시 조례1길 24 (조례동)
주소를 찾을 수 없습니다: 전북특별자치도 김제시 서암4길 45 (서암동)
주소를 찾을 수 없습니다: 전라남도 보성군 미력면 가평길 36-17


주소를 찾을 수 없습니다: 경기도 부천시 오정구 소사로 726, 지하4층~지상5층 (원종동)
주소를 찾을 수 없습니다: 경상북도 상주시 상서문로 53 (남성동)
주소를 찾을 수 없습니다: 경상남도 함안군 칠원읍 용산2길 45-13


주소를 찾을 수 없습니다: 경기도 가평군 설악면 미사리로 267-177


주소를 찾을 수 없습니다: 전라남도 영암군 영암읍 오리정길 8
주소를 찾을 수 없습니다: 충청남도 예산군 예산읍 신례원로 26
주소를 찾을 수 없습니다: 경상북도 김천시 신음1길 12 (신음동)
주소를 찾을 수 없습니다: 전라남도 보성군 벌교읍 남하로 12
주소를 찾을 수 없습니다: 전라남도 신안군 비금면 송치길 155-11
주소를 찾을 수 없습니다: 충청북도 음성군 금왕읍 음성로1230번길 10
주소를 찾을 수 없습니다: 전라남도 순천시 우명길 42 (연향동)


주소를 찾을 수 없습니다: 전라남도 장성군 장성읍 역전로 171
주소를 찾을 수 없습니다: 경상남도 창원시 의창구 용동로57번길 8 (사림동)
주소를 찾을 수 없습니다: 강원특별자치도 철원군 갈말읍 명성로 208-0
주소를 찾을 수 없습니다: 충청북도 청주시 서원구 1순환로 776-0 (개신동,충북대학교병원)


주소를 찾을 수 없습니다: 경상남도 창녕군 창녕읍 교리1길 2
주소를 찾을 수 없습니다: 경기도 파주시 탄현면 평화로574번길 17, (의)청학의료재단 파주성모병원.요양병원
주소를 찾을 수 없습니다: 광주광역시 광산구 용아로 259 하남성심병원 (산정동)
주소를 찾을 수 없습니다: 울산광역시 동구 대학병원로 25, 울산대학교병원 (전하동)
주소를 찾을 수 없습니다: 강원특별자치도 홍천군 홍천읍 산림공원1길 17
                 기관명                                            기관주소  \
0    (의)내경의료재단울산제일병원                      울산광역시 남구 남산로354번길 26 (신정동)   
1      (의)서일의료재단기장병원                         부산광역시 기장군 기장읍 대청로72번길 6   
2    (의)성세의료재단 뉴성민병원  인천광역시 서구 칠천왕로33번길 17 (석남동, 신석로 70(석남1동, 성민병원))   
3     (의)영문의료재단다보스병원       경기도 용인시 처인구 백옥대로1082번길 18, 다보스종합병원 (김량장동)   
4    (의)효심의료재단용인서울병원                        경기도 용인시 처인구 고림로 81 (고림동)   
..               ...                                             ...   
521      효산의료재단안양샘병원                  경기도 안양시 만안구 삼덕로 9 (안양동, 안양샘병원)   
522       효산의료재단지샘병원               경기도 군포시 군포로 591 (당동, (G샘병원)군포샘병원)   
523           효성시티병원                       부산광역시 해운대구 해운대로 135 (재송동)   
524             흑룡의원                           인

In [ ]:
df_basic

,기관명,기관주소,위도,경도
0,(의)내경의료재단울산제일병원,울산광역시 남구 남산로354번길 26 (신정동),35.548538,129.305706
1,(의)서일의료재단기장병원,부산광역시 기장군 기장읍 대청로72번길 6,35.234664,129.215478
2,(의)성세의료재단 뉴성민병원,"인천광역시 서구 칠천왕로33번길 17 (석남동, 신석로 70(석남1동, 성민병원))",37.509355,126.671129
3,(의)영문의료재단다보스병원,"경기도 용인시 처인구 백옥대로1082번길 18, 다보스종합병원 (김량장동)",NaN,NaN
4,(의)효심의료재단용인서울병원,경기도 용인시 처인구 고림로 81 (고림동),37.245068,127.218638
...,...,...,...,...
521,효산의료재단안양샘병원,"경기도 안양시 만안구 삼덕로 9 (안양동, 안양샘병원)",37.397804,126.915755
522,효산의료재단지샘병원,"경기도 군포시 군포로 591 (당동, (G샘병원)군포샘병원)",37.372789,126.942672
523,효성시티병원,부산광역시 해운대구 해운대로 135 (재송동),35.177160,129.186879
524,흑룡의원,인천광역시 옹진군 백령면 백령로 831,37.974239,124.694594


In [ ]:
# 응급실 데이터 수집하기
# 응급실 기본정보 조회

url = 'http://apis.data.go.kr/B552657/ErmctInfoInqireService/getEgytBassInfoInqire'
serviceKey = ''    # 여러분의 일반 인증키(Decoding)

params = {
    'serviceKey': serviceKey,
    'pageNo': '1', 'numOfRows': '1000',  # 전체 응급실 수가 500여개 됨. 1000개면 충분
    'format': 'xml'
}

response = requests.get(url, params = params)

# 정상 수행 되었다면 200
print(response)

<Response [200]>


In [ ]:
# response xml에서 주요 정보 찾기
root = ET.fromstring(response.text)

data = []

for item in root. findall('.//item'):
    duty_name = item.findtext('dutyName') #기관명
    duty_addr = item.findtext('dutyAddr') #주소
    # 필요한 정보 추가
    latitude = item.findtext('wgs84Lat')
    longitude = item.findtext('wgs84Lon')
    hvec = item.findtext('hvec')

    # 빈 리스트 data에 딕시너리 형태({'칼럼이름':값, ...})로 저장(추가)
    data.append({'기관명' : duty_name, '기관주소': duty_addr, '위도' : latitude, '경도': longitude, '응급실':hvec})

# 데이터프레임으로 변환
df_em =  pd.DataFrame(data)

In [ ]:
df_em

,기관명,기관주소,위도,경도,응급실
0,경희대학교병원,서울특별시 동대문구 경희대로 23 (회기동),37.5938765502235,127.05183223390303,16
1,건국대학교병원,서울특별시 광진구 능동로 120-1 (화양동),37.54084479467721,127.0721229093036,6
2,중앙대학교병원,서울특별시 동작구 흑석로 102 (흑석동),37.50707428493414,126.96079378447554,15
3,순천향대학교 부속 서울병원,서울특별시 용산구 대사관로 59 (한남동),37.53384172231443,127.00441798640304,20
4,이화여자대학교의과대학부속목동병원,서울특별시 양천구 안양천로 1071 (목동),37.53654282637804,126.8862159683056,30
...,...,...,...,...,...
995,수연세안과의원,"서울특별시 서초구 서초대로77길 54, 서초더블유타워 5, 6층 (서초동)",37.502506415764955,127.02484127797528,None
996,서초좋은의원,"서울특별시 서초구 서초중앙로 238, 306호 (반포동, 삼호가든상가)",37.5029619493601,127.012070984447,None
997,베스탑비뇨기과의원,"서울특별시 송파구 올림픽로 269, 2층 221호 (신천동, 롯데캐슬골드)",37.5144273491608,127.100611434725,None
998,연세이김마취통증의학과의원,서울특별시 송파구 송파대로30길 5 (가락동),37.4944808900047,127.118048260052,None


In [ ]:
df_merged = pd.merge(df_basic, df_em, on=['기관명', '기관주소'], how='inner')
df_merged

,기관명,기관주소,위도,경도,응급실
0,가톨릭대학교여의도성모병원,"서울특별시 영등포구 63로 10, 여의도성모병원 (여의도동)",37.51827233800711,126.93673129599131,17
1,강남고려병원,서울특별시 관악구 관악로 242 (봉천동),37.4856188363947,126.956781703053,10
2,강남베드로병원,"서울특별시 강남구 남부순환로 2649, 베드로병원 (도곡동)",37.485612179925724,127.0395873429168,None
3,강남힐병원,"서울특별시 관악구 남부순환로 1449, 강남힐병원 (신림동)",37.4816513439454,126.911648078471,None
4,강동경희대학교의대병원,서울특별시 강동구 동남로 892 (상일동),37.5520459324005,127.157084787845,12
...,...,...,...,...,...
57,한림대학교한강성심병원,"서울특별시 영등포구 버드나루로7길 12 (영등포동7가, 한강성심병원)",37.52346674579277,126.91033000120589,30
58,한양대학교병원,서울특별시 성동구 왕십리로 222-1 (사근동),37.559944533564746,127.04488284061982,10
59,혜민병원,서울특별시 광진구 자양로 85 (자양동),37.535315660180416,127.08360130258502,10
60,홍익병원,"서울특별시 양천구 목동로 225, 홍익병원본관 (신정동)",37.52844147447355,126.8636640030062,8


In [ ]:
# csv 파일로 저장(인덱스 제외)
df_basic.to_csv('df_basic_true.csv', index=False )

In [ ]:
# 응급실 데이터 수집하기
# 응급실 기본정보 조회

url = 'http://apis.data.go.kr/B552657/ErmctInfoInqireService/getEmrrmRltmUsefulSckbdInfoInqire'
serviceKey = ''    # 여러분의 일반 인증키(Decoding)

params = {
    'serviceKey': serviceKey,
    'pageNo': '1', 'numOfRows': '1000',  # 전체 응급실 수가 500여개 됨. 1000개면 충분
    'STAGE1' : '서울특별시', 'STAGE2':'강남구',
    'format': 'xml'
}

response = requests.get(url, params = params)

# 정상 수행 되었다면 200
print(response)

<Response [200]>


In [ ]:
response.text

In [ ]:
# response xml에서 주요 정보 찾기
root = ET.fromstring(response.text)

data = []

for item in root. findall('.//item'):
    duty_name = item.findtext('dutyName') #기관명
    # 필요한 정보 추가
    hvgc = item.findtext('hvgc') #입원실
    hvec = item.findtext('hvec') #응급실
    hvidate = item.findtext('hvidate') #입력일시
    # 빈 리스트 data에 딕시너리 형태({'칼럼이름':값, ...})로 저장(추가)
    data.append({'기관명' : duty_name, '응급실': hvec, '입원실': hvgc, '갱신일자': hvidate})

# 데이터프레임으로 변환
df_em =  pd.DataFrame(data)

In [ ]:
df_em

,기관명,응급실,입원실,갱신일자
0,연세대학교의과대학 강남세브란스병원,-6,127,20241118160826
1,삼성서울병원,-8,482,20241118161110
